# IC-1123 — Case study: which of 40 factors do we keep?

**ML & FinTech · 115-1 · 20261123 · Session 3 · work in pairs · 40 minutes, then discussion**

| Step | Min | What you do |
|---|---|---|
| 0 | 3 | Sit with a partner. Fill in both IDs; your target stock and 40 candidate factors are generated |
| 1 | 8 | Count the models **by hand** |
| 2 | 14 | Reproduce the analyst's R²; then do the selection honestly |
| 3 | 10 | The lasso; compare with your partner |
| 4 | 5 | Hand in **one sheet of paper per pair** |

You may use any AI tool for the code. The graded part is the paper your pair hands in, and what
you say when called on. Your numbers come from your 學號; your partner's differ.

---

## The case

A quant analyst pitches a factor model for your client's stock:

> *"We screened **40 candidate factors** and kept the 5 most correlated with the stock. The
> 5-factor model has an in-sample **R² that speaks for itself** (yours is printed in Step 2).
> We also cross-validated the 5-factor model and it holds up."*

Your client asks: **"Are these five real? Would the model work next year?"**

Your target stock and your 40 candidates (F01–F40) come from your 學號. Some candidates are real
market series and some are pure noise; you are not told which.

## Setup

In [ ]:
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import warnings; warnings.filterwarnings("ignore")

In [ ]:
STUDENT_ID = ""    # your 學號, in quotes, e.g. "315707999"
PARTNER_ID = ""    # your partner's 學號

In [ ]:
URL  = ("https://raw.githubusercontent.com/202609-ML-FinTech/00-course-info/"
        "main/in-class-exercise/historical_stock_prices_2017.csv")
CAND = ["historical_stock_prices_2017.csv", "../historical_stock_prices_2017.csv",
        "../00-course-info/in-class-exercise/historical_stock_prices_2017.csv"]
path = next((p for p in CAND if Path(p).exists()), URL)
prices = pd.read_csv(path, index_col=0).dropna(axis=1); prices.index = pd.to_datetime(prices.index)
ret = np.log(prices).diff().dropna() * 100                # daily log returns, %

def draw_portfolio(student_id, k=10):
    key = lambda t: hashlib.sha256(f"{int(student_id)}-{t}".encode()).hexdigest()
    return sorted(sorted(prices.columns, key=key)[:k])

assert str(STUDENT_ID).strip().isdigit(), "Fill in STUDENT_ID in the cell above, then run again."
MY_PORTFOLIO = draw_portfolio(STUDENT_ID)
print("Client of", STUDENT_ID, "holds:", MY_PORTFOLIO)
TARGET = MY_PORTFOLIO[0]
y = ret[TARGET]
others = [t for t in sorted(prices.columns) if t not in MY_PORTFOLIO][:20]
rng = np.random.default_rng(int(hashlib.sha256(f"{int(STUDENT_ID)}-noise".encode()).hexdigest(), 16) % 2**32)
X = ret[others].copy()
for j in range(20):
    X[f"noise{j}"] = rng.normal(0, 1, len(X))
X = X.iloc[:, rng.permutation(40)]                      # shuffled, then anonymised: the position tells you nothing
X.columns = [f"F{j+1:02d}" for j in range(40)]
print(f"Target: {TARGET}   candidates: {X.shape[1]} factors, {len(X)} days")

---
# Step 1 — Count the models by hand (8 min)

**Q1.** Best-subset selection over p = 40 factors fits how many models in total? 2^40 = ____ (about).

**Q2.** How many 5-factor models are there? C(40, 5) = 40! / (5! 35!) = ____ .
If each takes a millisecond, how long does best subset of size 5 take? ____

---
# Step 2 — The analyst's R², then the honest one (14 min)

## 2a. Reproduce (6 min)
Pick the 5 factors with the largest |correlation| with `y` on the **full sample**
(`X.corrwith(y).abs().sort_values(ascending=False).index[:5]`). Fit `LinearRegression` on them.
Print the five names and the in-sample R².

In [ ]:
# 2a: top-5 by full-sample correlation; in-sample R2



**Q3.** My five: ____ ; in-sample R² = ____ .

## 2b. Cross-validate the way the analyst did, then the honest way (8 min)
Use `KFold(5, shuffle=True, random_state=1)`.
- **Analyst's way:** the five factors are fixed (chosen on the full sample); compute the mean test R² across folds.
- **Honest way:** inside each fold, choose the top 5 on the training part only, fit, then test. Mean test R².

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold
# 2b: CV with selection outside the folds vs inside the folds



**Q4.** In-sample R² ____ ; CV R² with fixed factors ____ ; CV R² with selection inside folds ____ .

**Q5.** Three numbers for one model. Which one answers "would it work next year", and why do the other two look better?

> _replace this line_

---
# Step 3 — Let the lasso choose (10 min)

Fit `LassoCV(cv=5, random_state=0)` on all 40 standardised factors. Which factors keep a non-zero
coefficient? How many of the analyst's five survive?

In [ ]:
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler
# 3: LassoCV; non-zero coefficients



**Q6.** The lasso keeps ____ factors; ____ of the analyst's five survive.

**Q7.** Swap with your partner.

| | in-sample R² | honest CV R² | lasso keeps |
|---|---|---|---|
| me | | | |
| my partner | | | |

Different stocks, same story? What do you tell the client about the five factors?

> _replace this line_

---
# Step 4 — Hand in, on paper (5 min)

**One sheet per pair.** Write:

1. Both names and both 學號.
2. For **each** of you: in-sample R², honest CV R², how many factors the lasso keeps.
3. **Two sentences, agreed between you**, answering the client's question and the trap.

Give the sheet to Prof. Teng or Claire before you leave. **The sheet is the graded part.**

Then: *Restart & Run All*, and push this notebook as `in-class-exercise/1123/IC-1123.ipynb`
in your own repository before **12:10**.